# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kalyan-1845/flyrank-ml-assignment/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
print("""
My Rule: A simple heuristic that looks at the ratio of impressions between March and February. If March impressions are less than 80% of February impressions, AND February had at least 100 impressions, we flag the page as 'Needs Refresh'.

Reason Codes:
- SHARP_DECLINE: Impressions dropped by > 50%
- SLOW_BLEED: Impressions dropped by 20% to 50%
""")


My Rule: A simple heuristic that looks at the ratio of impressions between March and February. If March impressions are less than 80% of February impressions, AND February had at least 100 impressions, we flag the page as 'Needs Refresh'.

Reason Codes:
- SHARP_DECLINE: Impressions dropped by > 50%
- SLOW_BLEED: Impressions dropped by 20% to 50%



## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import duckdb
from google.colab import userdata
import pandas as pd
import os

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

print("Running baseline heuristic rule...")
query = f"""
    WITH monthly_data AS (
        SELECT content_hash_id,
               SUM(CASE WHEN report_date > '2026-02-28' AND report_date <= '2026-03-31' THEN gsc_impressions ELSE 0 END) AS imp_march,
               SUM(CASE WHEN report_date > '2026-01-31' AND report_date <= '2026-02-28' THEN gsc_impressions ELSE 0 END) AS imp_february
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-0*/*.parquet')
        GROUP BY content_hash_id
        HAVING imp_february >= 100
    )
    SELECT content_hash_id,
           imp_february,
           imp_march,
           (imp_march * 1.0 / imp_february) as retention_ratio,
           CASE
               WHEN (imp_march * 1.0 / imp_february) <= 0.5 THEN 'SHARP_DECLINE'
               ELSE 'SLOW_BLEED'
           END as reason_code
    FROM monthly_data
    WHERE (imp_march * 1.0 / imp_february) <= 0.8
    ORDER BY (imp_march - imp_february) ASC
    LIMIT 20
"""
df = con.sql(query).df()

# Create directory and save to CSV
os.makedirs('work/outputs', exist_ok=True)
df.to_csv('work/outputs/baseline_action_score.csv', index=False)

print("Top 5 Results:")
print(df.head())
print("\nQueue saved to work/outputs/baseline_action_score.csv!")

Running baseline heuristic rule...


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
print("""
For the Top 20 pages in the queue:
Action: 'Content Refresh / Audit'
Reason Code: Mostly SHARP_DECLINE because we ordered by the largest absolute drop in impressions.
Confidence Note: High confidence that traffic has indeed dropped, but LOW confidence on WHY.
What would make it wrong: If the page was about a seasonal event (like 'Valentine's Day Gifts'), a massive drop in March is perfectly normal. Refreshing it won't help.
""")

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
print("""
Weak Picks: Any page in the Top 20 that is tied to a specific date or one-time viral news event.
Leakage Check: Passed. This baseline rule only uses historical data (Feb/March) to make decisions. It does not peek into April's data to decide what to do.
""")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.